In [1]:
import polars as pl
from tqdm import tqdm

In [2]:
articles_path='../data/articles.parquet'
customer_path='../data/customers.parquet'
transaction_path='../data/transactions.parquet'

In [3]:
articles=pl.read_parquet(articles_path)
transactions=pl.read_parquet(transaction_path)

In [4]:
transactions['customer_id'].unique().shape

(1362281,)

In [5]:
DAY = 86400

In [6]:
def build_popular_items(transactions,window_days=7):
    max_time=transactions.select(pl.col('time').max()).item()
    start_time=max_time-window_days*DAY
    df=(
        transactions
        .filter(pl.col('time')>=start_time)
        .with_columns(
            weight=1/(1+(max_time-pl.col('time'))/DAY)
        )
        .group_by('article_id')
        .agg(
            score=pl.sum('weight'),
            cnt=pl.count()
        )
        .sort('score',descending=True)
    )
    return df

In [7]:
def recall_popularity(data,window_days,topk=50):
    df_pop=build_popular_items(data,window_days)
    top_item=df_pop.select('article_id').head(topk)['article_id'].to_list()

    users=data['customer_id'].unique()

    customer_ids = []
    article_ids = []
    ranks = []
    for cid in tqdm(users):
        for rank,aid in enumerate(top_item):
            customer_ids.append(cid)
            article_ids.append(aid)
            ranks.append(rank)
    del top_item,df_pop

    df = pl.DataFrame({
        "customer_id": customer_ids,
        "article_id": article_ids,
        "rank": ranks,
    }).with_columns(
        pl.col("rank").cast(pl.UInt8)
    )

    return df

In [13]:
res=recall_popularity(transactions,7)

/tmp/ipykernel_46403/715296805.py:13: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  cnt=pl.count()
100%|██████████| 1362281/1362281 [00:04<00:00, 288615.19it/s]


ok


In [22]:
res

customer_id,article_id,rank
str,i64,u8
"""9d110fdd2115a289b79e0b814ace58…",924243002,0
"""9d110fdd2115a289b79e0b814ace58…",924243001,1
"""9d110fdd2115a289b79e0b814ace58…",918522001,2
"""9d110fdd2115a289b79e0b814ace58…",866731001,3
"""9d110fdd2115a289b79e0b814ace58…",751471001,4
…,…,…
"""2aa44accf39c0b03832f8069c72e13…",877769001,45
"""2aa44accf39c0b03832f8069c72e13…",158340001,46
"""2aa44accf39c0b03832f8069c72e13…",874754002,47


In [14]:
res.to_pandas().info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 68114050 entries, 0 to 68114049
Data columns (total 3 columns):
 #   Column       Dtype 
---  ------       ----- 
 0   customer_id  object
 1   article_id   int64 
 2   rank         uint8 
dtypes: int64(1), object(1), uint8(1)
memory usage: 1.1+ GB


In [17]:
def get_validation_data(data: pl.DataFrame):
    DAY = 86400
    WEEK = 7 * DAY

    max_time = data.select(pl.col("time").max()).item()

    valid_start = max_time - 6 * DAY
    train_start = valid_start - 6 * WEEK

    train_df = data.filter(
        (pl.col("time") >= train_start) &
        (pl.col("time") <  valid_start)
    )

    valid_df = data.filter(
        pl.col("time") >= valid_start
    )

    return train_df, valid_df

In [32]:
def metric_recall(data,window_day=7,topk=5):
    train_df,valid_df=get_validation_data(data)

    user_item=recall_popularity(train_df,window_day,topk*10)
    user_set=set(user_item['customer_id'])

    pred_df = (
        user_item
        .sort("rank")
        .group_by("customer_id")
        .agg(pl.col("article_id").alias("pred_items"))
    )

    true_df = (
        valid_df
    .group_by("customer_id")
        .agg(pl.col('article_id').unique().alias("true_items")))

    eval_df=pred_df.join(true_df,on='customer_id',how='inner')

    for k in range(10, topk * 10 + 1, 10):
        total = 0.0
        cnt = 0

        for pred, true in zip(eval_df["pred_items"], eval_df["true_items"]):
            if len(true) == 0:
                continue
            total += len(set(pred[:k]) & set(true)) / len(true)
            cnt += 1

        print(f"Recall@{k}: {total / cnt:.6f}")

In [33]:
metric_recall(transactions)

/tmp/ipykernel_46403/715296805.py:13: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  cnt=pl.count()
100%|██████████| 312215/312215 [00:01<00:00, 274121.19it/s]


ok
Recall@10: 0.023949
Recall@20: 0.040023
Recall@30: 0.054456
Recall@40: 0.064255
Recall@50: 0.075289


In [35]:
metric_recall(transactions,window_day=14)

/tmp/ipykernel_46403/715296805.py:13: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  cnt=pl.count()
100%|██████████| 312215/312215 [00:01<00:00, 280216.99it/s]


Recall@10: 0.024542
Recall@20: 0.042435
Recall@30: 0.054520
Recall@40: 0.066011
Recall@50: 0.074562


In [36]:
metric_recall(transactions,window_day=28)

/tmp/ipykernel_46403/715296805.py:13: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  cnt=pl.count()
100%|██████████| 312215/312215 [00:01<00:00, 276547.80it/s]


Recall@10: 0.024205
Recall@20: 0.040233
Recall@30: 0.053620
Recall@40: 0.063694
Recall@50: 0.074615


In [8]:
def preserve_popularity_result(week=1):
    for w in range(1,week+1):
        res=recall_popularity(transactions,window_days=7*week)
        res.write_parquet(f'../save/candidate/recall_popularity_week{w}.parquet')

In [9]:
preserve_popularity_result(4)

/tmp/ipykernel_53853/715296805.py:13: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  cnt=pl.count()
100%|██████████| 1362281/1362281 [00:05<00:00, 270258.51it/s]
